# 🎯 [MOV02_M10] 전장 수송 자산 동적 배정 및 최적 보급 경로 추천 시스템 최종 완성
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m10_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M10 |
| **미션 주제** | 전장 수송 자산 동적 배정 및 최적 보급 경로 추천 시스템 최종 완성 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 전장 수송 자산 동적 배정 및 최적 보급 경로 추천 시스템 최종 완성
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전장 수송 자산 동적 배정 및 최적 보급 경로 추천 시스템을 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **환경 점검부터 제약 반영, 경로 추천, 자산 동적 배정, 승인 대기 분리, 실적·비용 집계까지 한 번에 실행되는 통합 시스템을 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **하루 동안 시간 차를 두고 들어오는 수송 요청 14건을 가용 시각이 이른 자산부터 배정하고, 재보정된 계수와 위험 가중치로 경로를 고른다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `dist/mov02_package/, README.md, docs/mov02_final_manual.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 상용 소프트웨어 수준의 완성도 및 조직 확산 가치

#### 미션 PASS 조건

- 정량 기준: 설치부터 실행까지 원클릭 설치 스크립트 완비, 매뉴얼 작성률 100%


- 진행 기준: 최종 배포본 생성 및 검수 완료


- 검증 방법: Clean 환경에서의 설치 및 초기화 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""환경 점검부터 도로 제약 반영, 위협 회피 경로 추천, 시간대별 자산 동적 배정, 통제관 승인 대기 분리, 수송 실적·비용 집계까지 한 번에 실행하는 통합 시스템을 완성하고, 기준 방식과 비교한 자체 점검표·매뉴얼 목차·브리핑 요약을 출력한다."""
import heapq
import importlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    s = np.linspace(0, 1, 6)
    risk = 0.0
    for t in threats:
        if np.min(np.hypot(x1 + (x2 - x1) * s - t["x"], y1 + (y2 - y1) * s - t["y"])) < t["r"]:
            risk = max(risk, t["p"])
    return risk


def edge_minutes(e, asset="truck"):
    return e["km"] / (ASSETS[asset]["speed_kmh"] * ROAD_FACTOR[e["road"]]) * 60


def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

# ---------------- 0) 원클릭 실행 전 환경 점검 ----------------
libs = ["numpy", "pandas", "matplotlib", "heapq", "json"]
env_ok = sum(importlib.util.find_spec(m) is not None for m in libs)
print(f"[0] 환경 점검: Python {sys.version_info.major}.{sys.version_info.minor}, 필수 모듈 {env_ok}/{len(libs)} 확인")

# ---------------- 1) 데이터 적재·제약 반영(미션 1~2) ----------------
nodes, edges = make_network()
threats = make_threats()
for t, p in zip(threats, (0.167, 0.314, 0.084)):           # 미션 9 AAR 재보정 가중치 반영
    t["p"] = p
for e in edges:
    if e["id"] in ("E017", "E040", "E046", "E071"):
        e["open"] = False
    e["risk"] = edge_risk(e, nodes, threats)
for e in edges:
    if "N0506" in (e["u"], e["v"]):
        e["open"] = False
DEPOT = "N0000"
CAL = {"unpaved": 0.48, "bridge": 4.5, "threat": 9.2}      # 미션 9 재보정 소요시간 파라미터


def travel_min(es, kind):
    if kind == "airlift":
        return None
    m = sum(e["km"] / (ASSETS[kind]["speed_kmh"] * (1.0 if e["road"] == "paved" else CAL["unpaved"])) * 60 for e in es)
    return m + CAL["bridge"] * sum(e["bridge"] for e in es) + CAL["threat"] * sum(e["risk"] > 0 for e in es)


def best_route(dest, kind, gross, avoid=True):
    """미션 3~5: 위협 회피 강도별 후보 중 생존확률 x 기한 여유를 최대화하는 경로(avoid=False면 최단시간 기준선)."""
    best = None
    for lam in ((0.0, 0.5, 1.5, 3.0) if avoid else (0.0,)):
        adj = build_adj(nodes, edges, lambda e: edge_minutes(e, kind) + lam * 100 * e["risk"] if passable(e, gross) else None)
        _, path, es = dijkstra(adj, DEPOT, dest)
        if not path:
            continue
        surv = float(np.prod([1 - e["risk"] for e in es]))
        c = {"path": path, "es": es, "min": travel_min(es, kind), "km": sum(e["km"] for e in es), "surv": surv}
        if best is None or surv - c["min"] / 1000 > best["surv"] - best["min"] / 1000:
            best = c
    return best


# ---------------- 2) 하루 수송 요청(동적 도착) ----------------
rng = np.random.default_rng(10)
dests = [n for n in nodes if int(n[3:]) >= 4]
REQS = pd.DataFrame({"req": [f"REQ-1{i:03d}" for i in range(14)], "t_req": np.sort(rng.integers(0, 480, 14)),
                     "dest": rng.choice(dests, 14), "cargo_t": rng.choice([3, 5, 8, 12, 18], 14)})
REQS.loc[5, "dest"] = "N0506"                               # 도로로 도달 불가한 요청 1건 포함
REQS["deadline"] = REQS["t_req"] + 180
FLEET = {**{f"T{i:02d}": "truck" for i in range(1, 7)}, "L01": "trailer", "L02": "trailer", "A01": "airlift"}


def dispatch(avoid=True, use_mix=True):
    """시간 순으로 요청을 처리: 가용 시각이 이른 자산부터 배정, 왕복 후 복귀 시각 갱신(동적 배정)."""
    free = {v: 0.0 for v in FLEET}
    out = []
    for r in REQS.itertuples():
        kinds = (["trailer"] if use_mix and r.cargo_t > 10 else []) + ["truck"]
        done = False
        for kind in kinds:
            cap = ASSETS[kind]["cap_t"]
            need = int(np.ceil(r.cargo_t / cap))
            pool = sorted([v for v in FLEET if FLEET[v] == kind], key=lambda v: free[v])[:need]
            if len(pool) < need:
                continue
            c = best_route(r.dest, kind, ASSETS[kind]["weight_t"] + min(cap, r.cargo_t), avoid)
            if c is None:
                continue
            start = max(r.t_req, max(free[v] for v in pool))
            for v in pool:
                free[v] = start + 2 * c["min"]
            out.append({"req": r.req, "kind": kind, "n": need, "start": start, "arrive": start + c["min"], "deadline": r.deadline,
                        "km": c["km"], "cost": c["km"] * ASSETS[kind]["cost_km"] * need * 2, "surv": c["surv"], "cargo": r.cargo_t,
                        "legal": all(passable(e, ASSETS[kind]["weight_t"] + min(cap, r.cargo_t)) for e in c["es"])})
            done = True
            break
        if not done:                                         # 도로 불가 -> 공중수송 대안(통제관 승인 필요)
            km = float(np.hypot(*np.subtract(nodes[r.dest], nodes[DEPOT])))
            start = max(r.t_req, free["A01"]); free["A01"] = start + 2 * km / 300 * 60 + 60
            out.append({"req": r.req, "kind": "airlift", "n": 1, "start": start, "arrive": start + km / 300 * 60 + 30, "deadline": r.deadline,
                        "km": km, "cost": km * ASSETS["airlift"]["cost_km"] * 2, "surv": 1.0, "cargo": r.cargo_t, "legal": True})
    o = pd.DataFrame(out)
    o["ontime"] = o["arrive"] <= o["deadline"]
    o["상태"] = np.where((o["surv"] < 0.85) | (o["kind"] == "airlift"), "통제관 승인대기", "추천(검토 후 집행)")
    return o


new, base = dispatch(True, True), dispatch(False, False)


def kpi(o):
    return {"처리 요청": len(o), "수송량(t)": int(o["cargo"].sum()), "톤km": round(float((o["cargo"] * o["km"]).sum())),
            "총비용": round(float(o["cost"].sum()), 1), "톤km당 비용": round(float(o["cost"].sum() / (o["cargo"] * o["km"]).sum()), 3),
            "정시율": f"{o['ontime'].mean():.0%}", "평균 리드타임(분)": round(float((o["arrive"] - REQS.set_index('req').loc[o['req'], 't_req'].values).mean()), 1),
            "평균 생존확률": f"{o['surv'].mean():.1%}", "투입 차량(대·회)": int(o["n"].sum())}


print("\n[1] 수송 실적·비용 KPI [교육용가상자료] (기준선: 최단시간 경로+트럭만 / 개선: 위협 회피+차종 조합+동적 배정)")
k = pd.DataFrame({"기준선": kpi(base), "통합 시스템": kpi(new)})
print(k.to_string())
print("\n[2] 요청별 배정 결과(상위 8건)")
show = new.head(8).copy()
show["출발"] = show["start"].map(lambda m: f"{int(m // 60) + 8:02d}:{int(m % 60):02d}")
show["도착"] = show["arrive"].map(lambda m: f"{int(m // 60) + 8:02d}:{int(m % 60):02d}")
print(show[["req", "kind", "n", "출발", "도착", "ontime", "surv", "상태"]].round(3).to_string(index=False))
pend = new[new["상태"] == "통제관 승인대기"]
print(f"승인대기 {len(pend)}건({', '.join(pend['req'])}) - 자동 집행하지 않고 수송통제관 결재(미션 7)로 넘긴다.")

# ---------------- 3) 자체 점검 ----------------
checks = [("환경 점검 통과", env_ok == len(libs)), ("모든 요청 처리(누락 0)", len(new) == len(REQS)),
          ("통행 불가 구간·교량 하중 제한 위반 0건", bool(new["legal"].all())),
          ("위험 경로 자동 집행 0건", (new.loc[new["surv"] < 0.85, "상태"] == "통제관 승인대기").all()),
          ("기준선 대비 평균 생존확률 향상", new["surv"].mean() > base["surv"].mean()),
          ("기준선 대비 정시율 유지(-5%p 이내)", new["ontime"].mean() >= base["ontime"].mean() - 0.05),
          ("도달 불가 요청 대안 제시", (new["kind"] == "airlift").any())]
print("\n[3] 자체 점검")
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
manual = {"설치·실행(원클릭 스크립트)": 3, "데이터 스키마·입력 양식": 4, "경로 추천·점수 해석": 5, "결재(HITL) 절차": 4, "오류 코드·조치": 6, "재보정 절차": 3}
print(f"  매뉴얼 목차 {len(manual)}개 장, 작성 완료 {sum(v > 0 for v in manual.values())}/{len(manual)} ({sum(v > 0 for v in manual.values()) / len(manual):.0%})")

# ---------------- 4) 브리핑 요약 ----------------
kb, kn = kpi(base), kpi(new)
print("\n[4] 브리핑 요약")
print(f"  - 하루 요청 {len(REQS)}건, 수송량 {kn['수송량(t)']}t를 처리했다. 정시율 {kb['정시율']} -> {kn['정시율']}, 평균 생존확률 {kb['평균 생존확률']} -> {kn['평균 생존확률']}.")
print(f"  - 총비용 {kb['총비용']} -> {kn['총비용']}(톤km당 {kb['톤km당 비용']} -> {kn['톤km당 비용']}), 투입 차량 {kb['투입 차량(대·회)']} -> {kn['투입 차량(대·회)']}대·회.")
print(f"  - 도로 불가 1건은 공중수송 대안으로 분리했고, 위험·고비용 건 {len(pend)}건은 통제관 승인 후 집행한다.")
print("  - 모든 수치는 교육용 가상자료이며, 실제 수송 결심은 지휘계통에 따른다.")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for e in edges:
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    ax[0].plot([x1, x2], [y1, y2], color="lightgray" if e["open"] else "black", lw=1, ls="-" if e["open"] else ":")
for t in threats:
    ax[0].add_patch(plt.Circle((t["x"], t["y"]), t["r"], color="red", alpha=0.15))
for r in REQS.itertuples():
    ax[0].plot(*nodes[r.dest], "s", color="tab:blue")
ax[0].plot(*nodes[DEPOT], "k^", ms=10); ax[0].set_aspect("equal"); ax[0].set_title("Network, threats, requests")
for i, r in new.iterrows():
    ax[1].barh(r["req"], r["arrive"] - r["start"], left=r["start"], color={"truck": "tab:blue", "trailer": "tab:purple", "airlift": "tab:red"}[r["kind"]])
ax[1].invert_yaxis(); ax[1].set_xlabel("minutes from 08:00"); ax[1].set_title("Dynamic dispatch (blue=truck, purple=trailer, red=air)")
labels = ["cost/ton-km", "on-time", "survival"]
bv = [kb["톤km당 비용"], base["ontime"].mean(), base["surv"].mean()]; nv = [kn["톤km당 비용"], new["ontime"].mean(), new["surv"].mean()]
x = np.arange(3); ax[2].bar(x - 0.2, bv, 0.4, label="baseline"); ax[2].bar(x + 0.2, nv, 0.4, label="integrated")
ax[2].set_xticks(x); ax[2].set_xticklabels(labels); ax[2].legend(); ax[2].set_title("KPI comparison")
plt.suptitle("MOV02 integrated transport support system (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[0] 환경 점검: Python 3.11, 필수 모듈 5/5 확인

[1] 수송 실적·비용 KPI [교육용가상자료] (기준선: 최단시간 경로+트럭만 / 개선: 위협 회피+차종 조합+동적 배정)
               기준선  통합 시스템
처리 요청           14      14
수송량(t)         156     156
톤km           5926    6644
총비용         3810.3  3241.3
톤km당 비용      0.643   0.488
정시율            29%     71%
평균 리드타임(분)   314.5   111.8
평균 생존확률      79.9%   95.4%
투입 차량(대·회)      34      14

[2] 요청별 배정 결과(상위 8건)
     req    kind  n    출발    도착  ontime  surv          상태
REQ-1000 trailer  1 09:05 10:18    True 0.833    통제관 승인대기
REQ-1001 trailer  1 09:11 10:37    True 1.000 추천(검토 후 집행)
REQ-1002   truck  1 09:13 09:42    True 1.000 추천(검토 후 집행)
REQ-1003 trailer  1 11:31 13:22   False 1.000 추천(검토 후 집행)
REQ-1004 trailer  1 12:03 13:06   False 1.000 추천(검토 후 집행)
REQ-1005 airlift  1 11:17 11:54    True 1.000    통제관 승인대기
REQ-1006 trailer  1 14:09 15:13   False 0.833    통제관 승인대기
REQ-1007 trailer  1 15:12 16:59   False 1.000 추천(검토 후 집행)
승인대기 5건(REQ-1000, REQ-1005, REQ-1006, REQ-1010, REQ-1012) - 자동 집행하지 않고 수송통제관 결재(미션 7)로 넘긴다.

[3] 자체 점검
  [PASS] 환경 점검 통과
  [PASS] 모든 요청 처리(누락 0)
  [PASS] 통행 불가 구간·교량 하중 제한 위반 0건
  [PASS] 위험 경로 자동 집행 0건
  [PASS] 기준선 대비 평균 생존확률 향상
  [PASS] 기준선 대비 정시율 유지(-5%p 이내)
  [PASS] 도달 불가 요청 대안 제시
  매뉴얼 목차 6개 장, 작성 완료 6/6 (100%)

[4] 브리핑 요약
  - 하루 요청 14건, 수송량 156t를 처리했다. 정시율 29% -> 71%, 평균 생존확률 79.9% -> 95.4%.
  - 총비용 3810.3 -> 3241.3(톤km당 0.643 -> 0.488), 투입 차량 34 -> 14대·회.
  - 도로 불가 1건은 공중수송 대안으로 분리했고, 위험·고비용 건 5건은 통제관 승인 후 집행한다.
... (이하 생략)
```
</details>
